# Clipt Detection Models — v4 Outcome Detection

Run cells **TOP TO BOTTOM** in order.  
Every model has its own download cell — run it immediately.  
Do **NOT** skip download cells.

| Section | Models |
|---------|--------|
| **SECTION 0** | Setup |
| **SECTION 1** | V4 Basketball (5 models) |
| **SECTION 2** | V4 Football (5 models) |
| **SECTION 3** | V4 Lacrosse (3 models) |
| **SECTION 4** | V4 Specialists (7 models) |
| **SECTION 5** | Final Summary |

**Total: 20 models**  
**Runtime: A100 GPU required**  
**Colab Secret: `ROBOFLOW_API_KEY`**

### IF COLAB DISCONNECTS
1. `colab.research.google.com` → Recent → `train_models_v4`
2. Runtime → Reconnect / Change to A100
3. Rerun **Setup** cell (0A) — it reinstalls packages
4. Rerun the **dataset download** cell for the current section
5. Continue from the last incomplete training cell
6. Already-downloaded `.pt` files are safe on your local machine

---
## SECTION 0: Setup

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0A — Install + Imports + GPU Check
# ═══════════════════════════════════════════════════════
!pip install roboflow ultralytics pyyaml -q

from google.colab import userdata, files
from roboflow import Roboflow
from ultralytics import YOLO
import os, torch, shutil, glob, yaml, time, traceback, gc

api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)

V4_BASE = "yolov8m.pt"
V4_IMGSZ = 832
V4_BATCH = 8
V4_DEVICE = 0 if torch.cuda.is_available() else "cpu"

assert torch.cuda.is_available(), "❌ NO GPU — go to Runtime → Change runtime type → A100"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✅ Roboflow API key loaded ({len(api_key)} chars)")

# Track training results for autopilot mode
TRAINING_LOG = []

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0B — download_model helper
# ═══════════════════════════════════════════════════════
def download_model(model_name, min_map50=0.4):
    """Validate, copy, and download a trained model."""
    base = model_name.replace('.pt', '')
    paths = sorted(glob.glob(f"runs/detect/{base}*/weights/best.pt"))
    path = paths[-1] if paths else None

    if not path:
        msg = f"❌ MISSING: {model_name} — no training run found"
        print(msg)
        available = glob.glob(f"runs/detect/{base}*")
        if available:
            print(f"   Found dirs: {available}")
        TRAINING_LOG.append((model_name, 'MISSING', 0, msg))
        return False

    try:
        metrics = YOLO(path).val()
        map50 = metrics.box.map50
    except Exception as e:
        # If validation fails, still download — model may work fine
        print(f"⚠️ {model_name} — val() failed: {e}")
        print(f"   Downloading anyway — check manually")
        shutil.copy(path, model_name)
        files.download(model_name)
        size_mb = os.path.getsize(path) / 1024 / 1024
        TRAINING_LOG.append((model_name, 'DOWNLOADED (no val)', 0, f'{size_mb:.1f}MB'))
        return True

    size_mb = os.path.getsize(path) / 1024 / 1024

    if map50 >= min_map50:
        shutil.copy(path, model_name)
        files.download(model_name)
        msg = f"✅ {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)"
        print(msg)
        TRAINING_LOG.append((model_name, 'PASS', map50, msg))
        return True
    else:
        msg = f"⚠️ {model_name} mAP50={map50:.3f} — below {min_map50} threshold"
        print(msg)
        print(f"   Downloading anyway — you decide whether to keep it")
        shutil.copy(path, model_name)
        files.download(model_name)
        TRAINING_LOG.append((model_name, 'LOW_MAP', map50, msg))
        return True

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0C — merge_datasets helper
# ═══════════════════════════════════════════════════════
def merge_datasets(dataset_a_path, dataset_b_path, merged_name):
    """Merge two YOLO datasets into one."""
    merged_dir = f"/content/{merged_name}"
    for split in ["train", "valid", "test"]:
        for subdir in ["images", "labels"]:
            os.makedirs(f"{merged_dir}/{split}/{subdir}", exist_ok=True)
            for prefix, path in [("a", dataset_a_path), ("b", dataset_b_path)]:
                src = f"{path}/{split}/{subdir}"
                dst = f"{merged_dir}/{split}/{subdir}"
                if os.path.exists(src):
                    for f in os.listdir(src):
                        shutil.copy(f"{src}/{f}", f"{dst}/{prefix}_{f}")

    # Use dataset_a's class config
    yaml_path = f"{dataset_a_path}/data.yaml"
    with open(yaml_path) as f:
        yaml_a = yaml.safe_load(f)

    merged_yaml = {
        'path': merged_dir,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': yaml_a['names'],
        'nc': yaml_a['nc']
    }
    with open(f"{merged_dir}/data.yaml", 'w') as f:
        yaml.dump(merged_yaml, f)

    count = len(os.listdir(f"{merged_dir}/train/images"))
    print(f"✅ Merged: {count} training images → {merged_dir}")
    return merged_dir

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0D — safe_train helper (autopilot crash recovery)
# ═══════════════════════════════════════════════════════
def safe_train(model_name, data_path, epochs, imgsz=V4_IMGSZ, batch=V4_BATCH, **kwargs):
    """
    Wraps YOLO training with crash recovery for autopilot mode.
    - Catches OOM errors → retries with half batch size
    - Catches any other error → logs it and continues
    - Clears GPU cache between attempts
    - Returns True on success, False on failure
    """
    if data_path is None:
        msg = f"⏭️ SKIPPED {model_name} — dataset unavailable"
        print(msg)
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, msg))
        return False

    # Resolve data path — could be a dataset object or a string
    if hasattr(data_path, 'location'):
        data_yaml = f"{data_path.location}/data.yaml"
    elif os.path.isdir(str(data_path)):
        data_yaml = f"{data_path}/data.yaml"
    else:
        data_yaml = str(data_path)

    if not os.path.exists(data_yaml):
        msg = f"❌ SKIPPED {model_name} — data.yaml not found at {data_yaml}"
        print(msg)
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, msg))
        return False

    attempts = [
        (batch, "full batch"),
        (max(batch // 2, 2), "half batch (OOM retry)"),
    ]

    for attempt_batch, label in attempts:
        try:
            print(f"\n{'='*60}")
            print(f"🚀 TRAINING: {model_name} ({label})")
            print(f"   data={data_yaml}")
            print(f"   epochs={epochs}, imgsz={imgsz}, batch={attempt_batch}")
            print(f"{'='*60}\n")

            torch.cuda.empty_cache()
            gc.collect()

            model = YOLO(V4_BASE)
            start = time.time()
            model.train(
                data=data_yaml,
                epochs=epochs,
                imgsz=imgsz,
                batch=attempt_batch,
                name=model_name.replace('.pt', ''),
                device=V4_DEVICE,
                **kwargs
            )
            elapsed = time.time() - start
            mins = elapsed / 60
            msg = f"✅ {model_name} complete in {mins:.1f} min"
            print(msg)
            TRAINING_LOG.append((model_name, 'TRAINED', 0, f'{mins:.1f} min'))
            return True

        except torch.cuda.OutOfMemoryError:
            print(f"\n⚠️ OOM on {model_name} with batch={attempt_batch}")
            torch.cuda.empty_cache()
            gc.collect()
            if attempt_batch == attempts[-1][0]:
                msg = f"❌ {model_name} — OOM even at batch={attempt_batch}"
                print(msg)
                TRAINING_LOG.append((model_name, 'OOM_FAIL', 0, msg))
                return False
            print(f"   Retrying with smaller batch...")

        except Exception as e:
            msg = f"❌ {model_name} — training error: {type(e).__name__}: {e}"
            print(msg)
            traceback.print_exc()
            TRAINING_LOG.append((model_name, 'ERROR', 0, msg))
            torch.cuda.empty_cache()
            gc.collect()
            return False

    return False

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0E — safe_download_dataset helper
# ═══════════════════════════════════════════════════════
def safe_download_dataset(workspace, project_name, version, var_name):
    """
    Download a Roboflow dataset with automatic version retry.
    Tries the requested version, then version-1, then version+1.
    Returns the dataset object or None.
    """
    versions_to_try = [version]
    if version > 1:
        versions_to_try.append(version - 1)
    versions_to_try.append(version + 1)

    for v in versions_to_try:
        try:
            project = rf.workspace(workspace).project(project_name)
            dataset = project.version(v).download("yolov8")
            print(f"✅ {var_name}: {dataset.location} (v{v})")
            return dataset
        except Exception as e:
            print(f"   ⚠️ {var_name} v{v} failed: {e}")

    print(f"❌ {var_name}: all versions failed")
    return None

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0F — Pre-flight dataset verification
# ═══════════════════════════════════════════════════════
print("🔍 Pre-flight: Verifying all Roboflow datasets are accessible...")
print("   This does NOT download — just checks the API responds.\n")

DATASETS_TO_VERIFY = [
    ("footballplayertracking", "jerseynumberdetectordigitdetector", 1, "Primary (jersey OCR)"),
    ("computer-vision-d5fjh", "basketball-detection-dn6fg", 1, "Basketball Hoop"),
    ("sc-xqmxu", "basketball-and-net-detection", 7, "Basketball Net"),
    ("roboflow-jvuqo", "basketball-player-detection-2", 1, "Basketball Actions"),
    ("zy-vevvi", "court-segmentation", 4, "Basketball Zones"),
    ("roboflow-universe-projects", "basketball-players-fy4c2", 1, "Basketball Players"),
    ("bronkscottema", "football-players-zm06l", 15, "Football Positions"),
    ("football-tracking", "football-presnap-tracker", 1, "Football Presnap"),
    ("augmented-startups", "football-player-detection-kucab", 1, "Football Players"),
    ("ryseai", "lacrosse-object-detection", 1, "Lacrosse"),
    ("rf100-vl", "lacrosse-object-detection-uxkt-vaybh", 1, "Lacrosse 2"),
]

verified = 0
failed = 0
for ws, proj, ver, label in DATASETS_TO_VERIFY:
    try:
        project = rf.workspace(ws).project(proj)
        _ = project.version(ver)
        print(f"  ✅ {label}: {ws}/{proj} v{ver}")
        verified += 1
    except Exception as e:
        # Try adjacent versions
        found = False
        for alt_v in [ver - 1, ver + 1, ver + 2]:
            if alt_v < 1:
                continue
            try:
                _ = project.version(alt_v)
                print(f"  ✅ {label}: {ws}/{proj} v{alt_v} (alt version)")
                verified += 1
                found = True
                break
            except:
                pass
        if not found:
            print(f"  ❌ {label}: {ws}/{proj} — FAILED")
            failed += 1

print(f"\n{'='*50}")
print(f"Pre-flight: {verified}/{len(DATASETS_TO_VERIFY)} verified, {failed} failed")
if failed > 0:
    print("⚠️ Some datasets failed — those models will be SKIPPED.")
    print("   Training will continue with available datasets.")
else:
    print("✅ All datasets accessible — ready to train!")
print(f"{'='*50}")

---
## SECTION 2: V4 Basketball

Outcome detection for basketball.  
Made shots, hoops, scoring zones, drives, rebounds.  
**5 models total.**

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2A — Download basketball datasets
# ═══════════════════════════════════════════════════════
if 'dataset_bball_hoop' not in dir() or dataset_bball_hoop is None:
    dataset_bball_hoop = safe_download_dataset(
        "computer-vision-d5fjh", "basketball-detection-dn6fg", 1, "dataset_bball_hoop")
else:
    print(f"✅ dataset_bball_hoop already loaded")

if 'dataset_bball_net' not in dir() or dataset_bball_net is None:
    dataset_bball_net = safe_download_dataset(
        "sc-xqmxu", "basketball-and-net-detection", 7, "dataset_bball_net")
else:
    print(f"✅ dataset_bball_net already loaded")

if 'dataset_bball_actions' not in dir() or dataset_bball_actions is None:
    dataset_bball_actions = safe_download_dataset(
        "roboflow-jvuqo", "basketball-player-detection-2", 1, "dataset_bball_actions")
else:
    print(f"✅ dataset_bball_actions already loaded")

if 'dataset_bball_zones' not in dir() or dataset_bball_zones is None:
    dataset_bball_zones = safe_download_dataset(
        "zy-vevvi", "court-segmentation", 4, "dataset_bball_zones")
else:
    print(f"✅ dataset_bball_zones already loaded")

if 'dataset_bball_players' not in dir() or dataset_bball_players is None:
    dataset_bball_players = safe_download_dataset(
        "roboflow-universe-projects", "basketball-players-fy4c2", 1, "dataset_bball_players")
else:
    print(f"✅ dataset_bball_players already loaded")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2B — Train basketball_hoop_detector_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "basketball_hoop_detector_v4.pt",
    dataset_bball_hoop,
    epochs=75,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

In [ ]:
# Cell 2C — Download basketball_hoop_detector_v4
download_model("basketball_hoop_detector_v4.pt", min_map50=0.5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2D — Train basketball_made_shot_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "basketball_made_shot_v4.pt",
    dataset_bball_net,
    epochs=75,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

In [ ]:
# Cell 2E — Download basketball_made_shot_v4
download_model("basketball_made_shot_v4.pt", min_map50=0.5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2F — Train basketball_scoring_zone_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "basketball_scoring_zone_v4.pt",
    dataset_bball_zones,
    epochs=75,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=1.0,
)

In [ ]:
# Cell 2G — Download basketball_scoring_zone_v4
download_model("basketball_scoring_zone_v4.pt", min_map50=0.5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2H — Train basketball_dribble_drive_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "basketball_dribble_drive_v4.pt",
    dataset_bball_actions,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=15,
    translate=0.2,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
)

In [ ]:
# Cell 2I — Download basketball_dribble_drive_v4
download_model("basketball_dribble_drive_v4.pt", min_map50=0.5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2J — Train basketball_rebound_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "basketball_rebound_v4.pt",
    dataset_bball_players,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=15,
    translate=0.2,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
)

In [ ]:
# Cell 2K — Download basketball_rebound_v4
download_model("basketball_rebound_v4.pt", min_map50=0.5)

---
## SECTION 3: V4 Football

Outcome detection for football.  
Completions, touchdowns, sacks, receptions, QB scrambles.  
**5 models total.**

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3A — Download football datasets
# ═══════════════════════════════════════════════════════
if 'dataset_fb_positions' not in dir() or dataset_fb_positions is None:
    dataset_fb_positions = safe_download_dataset(
        "bronkscottema", "football-players-zm06l", 15, "dataset_fb_positions")
else:
    print(f"✅ dataset_fb_positions already loaded")

if 'dataset_fb_presnap' not in dir() or dataset_fb_presnap is None:
    dataset_fb_presnap = safe_download_dataset(
        "football-tracking", "football-presnap-tracker", 1, "dataset_fb_presnap")
else:
    print(f"✅ dataset_fb_presnap already loaded")

if 'dataset_fb_players' not in dir() or dataset_fb_players is None:
    dataset_fb_players = safe_download_dataset(
        "augmented-startups", "football-player-detection-kucab", 1, "dataset_fb_players")
else:
    print(f"✅ dataset_fb_players already loaded")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3B — Train football_completion_detector_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "football_completion_detector_v4.pt",
    dataset_fb_positions,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=15,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

In [ ]:
# Cell 3C — Download football_completion_detector_v4
download_model("football_completion_detector_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3D — Train football_touchdown_detector_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "football_touchdown_detector_v4.pt",
    dataset_fb_presnap,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20,
    translate=0.2,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
)

In [ ]:
# Cell 3E — Download football_touchdown_detector_v4
download_model("football_touchdown_detector_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3F — Train football_sack_detector_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "football_sack_detector_v4.pt",
    dataset_fb_players,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
)

In [ ]:
# Cell 3G — Download football_sack_detector_v4
download_model("football_sack_detector_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3H — Train football_reception_yac_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "football_reception_yac_v4.pt",
    dataset_fb_positions,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.025,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.25,
    scale=0.7,
    fliplr=0.7,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
)

In [ ]:
# Cell 3I — Download football_reception_yac_v4
download_model("football_reception_yac_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3J — Train football_qb_scramble_v4 (merged dataset)
# ═══════════════════════════════════════════════════════
merged_fb_qb = None
if dataset_fb_positions is not None and dataset_fb_presnap is not None:
    merged_fb_qb = merge_datasets(
        dataset_fb_positions.location,
        dataset_fb_presnap.location,
        "merged_fb_qb_scramble"
    )
elif dataset_fb_positions is not None:
    print("⚠️ Only positions dataset available — using it alone")
    merged_fb_qb = dataset_fb_positions.location
elif dataset_fb_presnap is not None:
    print("⚠️ Only presnap dataset available — using it alone")
    merged_fb_qb = dataset_fb_presnap.location

safe_train(
    "football_qb_scramble_v4.pt",
    merged_fb_qb,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
)

In [ ]:
# Cell 3K — Download football_qb_scramble_v4
download_model("football_qb_scramble_v4.pt", min_map50=0.4)

---
## SECTION 4: V4 Lacrosse

Outcome detection for lacrosse.  
Goals, shot quality, ground balls.  
**3 models total.**

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4A — Download lacrosse datasets
# ═══════════════════════════════════════════════════════
if 'dataset_lax' not in dir() or dataset_lax is None:
    dataset_lax = safe_download_dataset(
        "ryseai", "lacrosse-object-detection", 1, "dataset_lax")
else:
    print(f"✅ dataset_lax already loaded")

if 'dataset_lax2' not in dir() or dataset_lax2 is None:
    dataset_lax2 = safe_download_dataset(
        "rf100-vl", "lacrosse-object-detection-uxkt-vaybh", 1, "dataset_lax2")
else:
    print(f"✅ dataset_lax2 already loaded")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4B — Train lacrosse_goal_detector_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "lacrosse_goal_detector_v4.pt",
    dataset_lax,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=15,
    translate=0.2,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
)

In [ ]:
# Cell 4C — Download lacrosse_goal_detector_v4
download_model("lacrosse_goal_detector_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4D — Train lacrosse_shot_quality_v4 (merged dataset)
# ═══════════════════════════════════════════════════════
merged_lax_shot = None
if dataset_lax is not None and dataset_lax2 is not None:
    merged_lax_shot = merge_datasets(
        dataset_lax.location,
        dataset_lax2.location,
        "merged_lax_shot_quality"
    )
elif dataset_lax is not None:
    print("⚠️ Only lax dataset available — using it alone")
    merged_lax_shot = dataset_lax.location
elif dataset_lax2 is not None:
    print("⚠️ Only lax2 dataset available — using it alone")
    merged_lax_shot = dataset_lax2.location

safe_train(
    "lacrosse_shot_quality_v4.pt",
    merged_lax_shot,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
)

In [ ]:
# Cell 4E — Download lacrosse_shot_quality_v4
download_model("lacrosse_shot_quality_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4F — Train lacrosse_ground_ball_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "lacrosse_ground_ball_v4.pt",
    dataset_lax,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=10,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.2,
    erasing=0.5,
)

In [ ]:
# Cell 4G — Download lacrosse_ground_ball_v4
download_model("lacrosse_ground_ball_v4.pt", min_map50=0.4)

---
## SECTION 5: V4 Specialists

Niche scenario coverage.  
Crowd energy, night games, indoor, obstructions,  
helmet glare, low resolution, multi-player clusters.  
**7 models total.**

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5A — Ensure primary dataset loaded + VERIFY
# ═══════════════════════════════════════════════════════
# If Cell 1A already ran, dataset_primary is loaded with merged splits.
# If resuming from Section 5, re-download + merge here.

if 'dataset_primary' not in dir() or dataset_primary is None:
    project = rf.workspace("footballplayertracking").project(
        "jerseynumberdetectordigitdetector"
    )
    dataset_primary = None
    best_total = 0
    for v in range(0, 11):
        try:
            ds = project.version(v).download("yolov8")
            loc = ds.location
            total = sum(
                len(os.listdir(f"{loc}/{s}/images"))
                for s in ["train", "valid", "test"]
                if os.path.isdir(f"{loc}/{s}/images")
            )
            print(f"  v{v}: {total} total images")
            if total > best_total:
                best_total = total
                dataset_primary = ds
        except:
            pass

    # Merge valid+test into train
    if dataset_primary is not None:
        loc = dataset_primary.location
        train_img = f"{loc}/train/images"
        train_lbl = f"{loc}/train/labels"
        merged = 0
        for split in ["valid", "test"]:
            src_img = f"{loc}/{split}/images"
            src_lbl = f"{loc}/{split}/labels"
            if os.path.isdir(src_img):
                for fname in os.listdir(src_img):
                    dst = f"{train_img}/{split}_{fname}"
                    if not os.path.exists(dst):
                        shutil.copy2(f"{src_img}/{fname}", dst)
                        merged += 1
            if os.path.isdir(src_lbl):
                for fname in os.listdir(src_lbl):
                    dst = f"{train_lbl}/{split}_{fname}"
                    if not os.path.exists(dst):
                        shutil.copy2(f"{src_lbl}/{fname}", dst)
        # Update data.yaml val path
        with open(f"{loc}/data.yaml") as f:
            data_cfg = yaml.safe_load(f)
        data_cfg['val'] = data_cfg.get('train', 'train/images')
        with open(f"{loc}/data.yaml", 'w') as f:
            yaml.dump(data_cfg, f)
        final_count = len(os.listdir(train_img))
        print(f"✅ Primary dataset ready: {final_count} training images (merged {merged} from valid+test)")
else:
    loc = dataset_primary.location
    final_count = len(os.listdir(f"{loc}/train/images"))
    print(f"✅ Primary already loaded: {loc} ({final_count} training images)")

# Verify
if dataset_primary is not None:
    with open(f"{dataset_primary.location}/data.yaml") as f:
        _verify = yaml.safe_load(f)
    _nc = _verify.get('nc', 0)
    _train_count = len(os.listdir(f"{dataset_primary.location}/train/images"))
    assert _nc == 1, f"WRONG DATASET — expected nc=1 got {_nc}"
    assert _train_count > 1000, f"Dataset too small: {_train_count} images"
    print(f"✅ Verified: nc={_nc}, {_train_count} training images")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5B — Search for crowd/stadium dataset
# ═══════════════════════════════════════════════════════
dataset_crowd = None

# Try known crowd/stadium datasets on Roboflow
crowd_candidates = [
    ("roboflow-universe-projects", "crowd-counting-thermal", 1),
    ("crowd-counting-acotd", "crowd-counting-6gxq7", 1),
    ("sam-uvt4u", "crowd-detection-yituf", 1),
]

for ws, proj, ver in crowd_candidates:
    try:
        dataset_crowd = safe_download_dataset(ws, proj, ver, "dataset_crowd")
        if dataset_crowd is not None:
            break
    except:
        continue

if dataset_crowd is None:
    print("⚠️ No crowd dataset found — will use dataset_primary as fallback")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5C — Train crowd_energy_detector_v4
# ═══════════════════════════════════════════════════════
crowd_data = dataset_crowd if dataset_crowd is not None else dataset_primary

safe_train(
    "crowd_energy_detector_v4.pt",
    crowd_data,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=15,
    translate=0.2,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
)

In [ ]:
# Cell 5D — Download crowd_energy_detector_v4
download_model("crowd_energy_detector_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5E — Train night_game_specialist_v4
# ═══════════════════════════════════════════════════════

assert 'dataset_primary' in dir() and dataset_primary is not None, \
    "❌ dataset_primary not loaded — run Cell 5A first!"
with open(f"{dataset_primary.location}/data.yaml") as f:
    _data = yaml.safe_load(f)
nc = _data.get('nc', 0)
train_count = len(os.listdir(f"{dataset_primary.location}/train/images"))
assert nc == 1, f"WRONG DATASET: nc={nc}, expected 1 — fix download cell"
print(f"Dataset: {dataset_primary.location}")
print(f"Classes: {nc}, Training images: {train_count}")

safe_train(
    "night_game_specialist_v4.pt",
    dataset_primary,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.03,
    hsv_s=0.9,
    hsv_v=0.9,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    erasing=0.4,
)

In [ ]:
# Cell 5F — Download night_game_specialist_v4
download_model("night_game_specialist_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5G — Prepare indoor dataset
# ═══════════════════════════════════════════════════════
# Use basketball hoop dataset if available (indoor courts),
# otherwise fall back to primary
indoor_data = None
if 'dataset_bball_hoop' in dir() and dataset_bball_hoop is not None:
    indoor_data = dataset_bball_hoop
    print(f"✅ Using basketball hoop dataset for indoor training")
else:
    indoor_data = dataset_primary
    print(f"⚠️ Using primary dataset as fallback for indoor")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5H — Train indoor_court_specialist_v4
# ═══════════════════════════════════════════════════════
safe_train(
    "indoor_court_specialist_v4.pt",
    indoor_data,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=10,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
)

In [ ]:
# Cell 5I — Download indoor_court_specialist_v4
download_model("indoor_court_specialist_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5J — Train crowd_obstruction_specialist_v4
# ═══════════════════════════════════════════════════════

assert 'dataset_primary' in dir() and dataset_primary is not None, \
    "❌ dataset_primary not loaded — run Cell 5A first!"
with open(f"{dataset_primary.location}/data.yaml") as f:
    _data = yaml.safe_load(f)
nc = _data.get('nc', 0)
train_count = len(os.listdir(f"{dataset_primary.location}/train/images"))
assert nc == 1, f"WRONG DATASET: nc={nc}, expected 1 — fix download cell"
print(f"Dataset: {dataset_primary.location}")
print(f"Classes: {nc}, Training images: {train_count}")

safe_train(
    "crowd_obstruction_specialist_v4.pt",
    dataset_primary,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.25,
    scale=0.8,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.3,
    copy_paste=0.6,
    erasing=0.7,
)

In [ ]:
# Cell 5K — Download crowd_obstruction_specialist_v4
download_model("crowd_obstruction_specialist_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5L — Train helmet_glare_specialist_v4
# ═══════════════════════════════════════════════════════

assert 'dataset_primary' in dir() and dataset_primary is not None, \
    "❌ dataset_primary not loaded — run Cell 5A first!"
with open(f"{dataset_primary.location}/data.yaml") as f:
    _data = yaml.safe_load(f)
nc = _data.get('nc', 0)
train_count = len(os.listdir(f"{dataset_primary.location}/train/images"))
assert nc == 1, f"WRONG DATASET: nc={nc}, expected 1 — fix download cell"
print(f"Dataset: {dataset_primary.location}")
print(f"Classes: {nc}, Training images: {train_count}")

safe_train(
    "helmet_glare_specialist_v4.pt",
    dataset_primary,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.03,
    hsv_s=0.9,
    hsv_v=0.9,
    degrees=25,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    erasing=0.5,
)

In [ ]:
# Cell 5M — Download helmet_glare_specialist_v4
download_model("helmet_glare_specialist_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5N — Train low_resolution_specialist_v4
# ═══════════════════════════════════════════════════════

assert 'dataset_primary' in dir() and dataset_primary is not None, \
    "❌ dataset_primary not loaded — run Cell 5A first!"
with open(f"{dataset_primary.location}/data.yaml") as f:
    _data = yaml.safe_load(f)
nc = _data.get('nc', 0)
train_count = len(os.listdir(f"{dataset_primary.location}/train/images"))
assert nc == 1, f"WRONG DATASET: nc={nc}, expected 1 — fix download cell"
print(f"Dataset: {dataset_primary.location}")
print(f"Classes: {nc}, Training images: {train_count}")

safe_train(
    "low_resolution_specialist_v4.pt",
    dataset_primary,
    epochs=150,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.7,
    degrees=15,
    translate=0.2,
    scale=0.8,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.3,
    copy_paste=0.3,
    erasing=0.5,
)

In [ ]:
# Cell 5O — Download low_resolution_specialist_v4
download_model("low_resolution_specialist_v4.pt", min_map50=0.4)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5P — Prepare cluster dataset (merge all sports)
# ═══════════════════════════════════════════════════════
dataset_cluster = None

# Collect available player datasets
cluster_datasets = []
if 'dataset_fb_players' in dir() and dataset_fb_players is not None:
    cluster_datasets.append(dataset_fb_players.location)
if 'dataset_bball_players' in dir() and dataset_bball_players is not None:
    cluster_datasets.append(dataset_bball_players.location)
if 'dataset_lax' in dir() and dataset_lax is not None:
    cluster_datasets.append(dataset_lax.location)

if len(cluster_datasets) >= 2:
    # Merge first two, then merge result with third if exists
    merged_path = merge_datasets(cluster_datasets[0], cluster_datasets[1], "merged_cluster_ab")
    if len(cluster_datasets) == 3:
        merged_path = merge_datasets(merged_path, cluster_datasets[2], "merged_cluster_abc")
    dataset_cluster = merged_path
    print(f"✅ Cluster dataset ready: {merged_path}")
elif len(cluster_datasets) == 1:
    dataset_cluster = cluster_datasets[0]
    print(f"⚠️ Only 1 player dataset — using it alone")
elif dataset_primary is not None:
    dataset_cluster = dataset_primary.location
    print(f"⚠️ No player datasets — using primary as fallback")
else:
    print(f"❌ No datasets available for cluster model")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5Q — Train multi_player_cluster_v4
# ═══════════════════════════════════════════════════════

# Show dataset info before training
if dataset_cluster is not None:
    _cluster_yaml = f"{dataset_cluster}/data.yaml" if isinstance(dataset_cluster, str) else f"{dataset_cluster.location}/data.yaml"
    if os.path.exists(_cluster_yaml):
        with open(_cluster_yaml) as f:
            _data = yaml.safe_load(f)
        _nc = _data.get('nc', 0)
        _cluster_path = dataset_cluster if isinstance(dataset_cluster, str) else dataset_cluster.location
        _train_dir = f"{_cluster_path}/train/images"
        _train_count = len(os.listdir(_train_dir)) if os.path.exists(_train_dir) else 0
        print(f"Dataset: {_cluster_path}")
        print(f"Classes: {_nc}, Training images: {_train_count}")

safe_train(
    "multi_player_cluster_v4.pt",
    dataset_cluster,
    epochs=100,
    imgsz=V4_IMGSZ,
    batch=V4_BATCH,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.3,
    copy_paste=0.5,
    erasing=0.4,
)

In [ ]:
# Cell 5R — Download multi_player_cluster_v4
download_model("multi_player_cluster_v4.pt", min_map50=0.4)

---
## SECTION 6: Final Summary

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6A — Complete training report
# ═══════════════════════════════════════════════════════
all_models = [
    ("basketball_hoop_detector_v4.pt", "V4 Basketball", 0.5),
    ("basketball_made_shot_v4.pt", "V4 Basketball", 0.5),
    ("basketball_scoring_zone_v4.pt", "V4 Basketball", 0.5),
    ("basketball_dribble_drive_v4.pt", "V4 Basketball", 0.5),
    ("basketball_rebound_v4.pt", "V4 Basketball", 0.5),
    ("football_completion_detector_v4.pt", "V4 Football", 0.4),
    ("football_touchdown_detector_v4.pt", "V4 Football", 0.4),
    ("football_sack_detector_v4.pt", "V4 Football", 0.4),
    ("football_reception_yac_v4.pt", "V4 Football", 0.4),
    ("football_qb_scramble_v4.pt", "V4 Football", 0.4),
    ("lacrosse_goal_detector_v4.pt", "V4 Lacrosse", 0.4),
    ("lacrosse_shot_quality_v4.pt", "V4 Lacrosse", 0.4),
    ("lacrosse_ground_ball_v4.pt", "V4 Lacrosse", 0.4),
    ("crowd_energy_detector_v4.pt", "V4 Specialist", 0.4),
    ("night_game_specialist_v4.pt", "V4 Specialist", 0.4),
    ("indoor_court_specialist_v4.pt", "V4 Specialist", 0.4),
    ("crowd_obstruction_specialist_v4.pt", "V4 Specialist", 0.4),
    ("helmet_glare_specialist_v4.pt", "V4 Specialist", 0.4),
    ("low_resolution_specialist_v4.pt", "V4 Specialist", 0.4),
    ("multi_player_cluster_v4.pt", "V4 Specialist", 0.4),
]

passed = []
trained_not_downloaded = []
missing = []

for name, section, threshold in all_models:
    base = name.replace('.pt', '')
    if os.path.exists(name):
        passed.append(f"✅ {name} ({section})")
    elif glob.glob(f"runs/detect/{base}*/weights/best.pt"):
        trained_not_downloaded.append(
            f"⚠️ {name} — trained, run download cell ({section})"
        )
    else:
        missing.append(f"❌ {name} — not trained ({section})")

print("=" * 60)
print(f"FINAL REPORT — {len(passed)}/20 MODELS DOWNLOADED")
print("=" * 60)

print(f"\n📦 DOWNLOADED ({len(passed)}/20):")
for m in passed:
    print(f"  {m}")

if trained_not_downloaded:
    print(f"\n⚠️ TRAINED BUT NOT DOWNLOADED ({len(trained_not_downloaded)}):")
    for m in trained_not_downloaded:
        print(f"  {m}")

if missing:
    print(f"\n❌ MISSING ({len(missing)}):")
    for m in missing:
        print(f"  {m}")

# Print training log from safe_train
if TRAINING_LOG:
    print(f"\n{'='*60}")
    print("TRAINING LOG (from this session):")
    print(f"{'='*60}")
    for name, status, score, detail in TRAINING_LOG:
        print(f"  {status:15s} | {name:45s} | {detail}")

print(f"\n{'='*60}")
print("NEXT STEPS:")
print("  1. Upload all .pt files to app/model/ in the repo")
print("  2. git add app/model/*.pt")
print('  3. git commit -m \"Add v4 trained models\"')
print("  4. git push")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6B — Autopilot debug: check for any issues
# ═══════════════════════════════════════════════════════
# Run this after all training to diagnose any problems.
# Safe to run — does NOT retrain anything.

print("🔍 AUTOPILOT DIAGNOSTICS")
print("=" * 60)

# 1. GPU health
print("\n[1/5] GPU STATUS:")
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated(0) / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  ✅ VRAM: {mem_used:.1f}/{mem_total:.1f} GB used")
else:
    print("  ❌ GPU not available — runtime may have disconnected")

# 2. Disk space
print("\n[2/5] DISK SPACE:")
stat = os.statvfs('/') if hasattr(os, 'statvfs') else None
if stat:
    free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
    total_gb = (stat.f_blocks * stat.f_frsize) / 1e9
    print(f"  {'✅' if free_gb > 5 else '⚠️'} Free: {free_gb:.1f}/{total_gb:.1f} GB")
    if free_gb < 5:
        print("  ⚠️ Low disk! Run: !rm -rf runs/detect/*/  (after downloading models)")
else:
    import subprocess
    result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
    print(f"  {result.stdout}")

# 3. Check all runs directories
print("\n[3/5] TRAINING RUNS FOUND:")
runs = sorted(glob.glob("runs/detect/*/"))
if runs:
    for r in runs:
        best = os.path.exists(f"{r}weights/best.pt")
        print(f"  {'✅' if best else '❌'} {r} {'(has best.pt)' if best else '(NO best.pt — crashed?)'}")
else:
    print("  ❌ No training runs found")

# 4. Check downloaded .pt files
print("\n[4/5] DOWNLOADED .PT FILES:")
pt_files = sorted(glob.glob("*.pt"))
for f in pt_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    if size_mb < 1:
        print(f"  ⚠️ {f} ({size_mb:.1f}MB) — suspiciously small!")
    else:
        print(f"  ✅ {f} ({size_mb:.1f}MB)")
if not pt_files:
    print("  ❌ No .pt files in working directory")

# 5. Errors from training log
print("\n[5/5] ERROR SUMMARY:")
errors = [x for x in TRAINING_LOG if x[1] in ('ERROR', 'OOM_FAIL', 'SKIPPED', 'MISSING')]
if errors:
    for name, status, _, detail in errors:
        print(f"  ❌ {name}: {detail}")
    print(f"\n  {len(errors)} issue(s) — scroll up to find the failing cell")
    print(f"  Fix the issue and rerun JUST that cell + its download cell")
else:
    print("  ✅ No errors recorded in training log")

print(f"\n{'='*60}")
total_downloaded = len([f for f in glob.glob("*.pt") if os.path.getsize(f) > 1024*1024])
print(f"SCORE: {total_downloaded}/20 models downloaded successfully")
if total_downloaded == 23:
    print("🎉 ALL MODELS COMPLETE — ready to push to repo!")
elif total_downloaded >= 20:
    print("👍 Almost there — check errors above and rerun failed cells")
else:
    print("⚠️ Several models missing — check errors above")
print(f"{'='*60}")